# Lesson 4: Awkward Array

NumPy-style arrays for events with different numbers of particles.

<img src="../img/map-4.svg" width="85%" alt="">

## The events we threw away

Projects 2 and 3 used 30,868 events, out of 299,973: only the events with exactly two electrons and two muons fit in flat NumPy arrays.

A real analysis cannot drop 90% of its data for the convenience of its arrays. Awkward Array keeps every event, whatever its shape, and keeps the NumPy moves you already know.

## Jagged events

Events have different numbers of particles, so they do not fit in a NumPy array.

<img src="../img/rect-vs-jagged.svg" width="85%" alt="">

In [ ]:
import numpy as np

np.array([[63.0, 38.1, 4.0], [], [54.3, 23.5]])

## Data in the wild is jagged

Not only particles: storms and their measurements, movies and their casts, books and their chapters. Any JSON file is lists inside records inside lists.

The rectangle is the exception, not the rule. What NumPy did for rectangles, Awkward Array does for everything else.

## Awkward Arrays

An Awkward Array is an array whose rows can have different lengths.

An array like this is called jagged.

<img src="../img/jagged.svg" width="70%" alt="">

In [ ]:
import awkward as ak

pt = ak.Array([[63.0, 38.1, 4.0], [], [54.3, 23.5], [41.2]])
pt

## How it fits in memory

No pointers, no Python lists: the values sit in one flat array, and a second array of offsets says where each event begins and ends.

That is why the NumPy speed survives the jaggedness.

In [ ]:
pt.layout

## Types

The type describes the structure. Read it from left to right.

<img src="../img/type.svg" width="70%" alt="">

In [ ]:
pt.type.show()

## Same as NumPy: arithmetic

Arithmetic and NumPy functions apply to every element.

In [ ]:
np.sqrt(pt * 2)

## New: `ak.num`

`ak.num` counts the items in each event.

In [ ]:
ak.num(pt)

## New: empty events

The first muon of every event fails when an event is empty.

Select the events that have a muon first, then `[:, 0]` is safe.

In [ ]:
pt[:, 0]

## New: two kinds of selection

There are two kinds of selection: drop particles or drop events.

<img src="../img/two-masks.svg" width="85%" alt="">

## Jagged masks

A jagged mask, one value per muon, drops particles and keeps every event.

<img src="../img/mask-particles.svg" width="75%" alt="">

In [ ]:
pt[pt > 20]

## Flat masks

A flat mask, one value per event, drops whole events.

<img src="../img/mask-events.svg" width="75%" alt="">

In [ ]:
pt[ak.num(pt) >= 2]

## New: reductions with `axis=1`

A reduction needs `axis=1` to give one value per event.

An empty event has no maximum, so its result is `None`.

<img src="../img/reduce-max.svg" width="75%" alt="">

In [ ]:
ak.max(pt, axis=1)

## Records

Records group the properties of one particle. A dot reads one field of every particle.

<img src="../img/record.svg" width="75%" alt="">

In [ ]:
charge = ak.Array([[1, -1, 1], [], [1, -1], [-1]])
muons = ak.zip({"pt": pt, "charge": charge})
muons.charge

## Real data

One array holds every event with every field.

In [ ]:
events = ak.from_parquet("../data/SMHiggsToZZTo4L.parquet")
events.fields

In [ ]:
events.muon.pt

## `ak.flatten`

`ak.flatten` removes the event structure, which a histogram needs.

<img src="../img/flatten.svg" width="70%" alt="">

In [ ]:
import hist

h = hist.Hist.new.Reg(50, 0, 100, label="muon pT [GeV]").Double()
h.fill(ak.flatten(events.muon.pt))
h.plot();

## `vector`

`vector` makes the muons four-vectors: `+` adds them and `.mass` gives the mass.

This is the `FourVector` class of lesson 3, for Awkward Arrays.

<img src="../img/two-muons.svg" width="60%" alt="">

In [ ]:
import vector

vector.register_awkward()
two = events.muon[ak.num(events.muon) == 2]
(two[:, 0] + two[:, 1]).mass

## `ak.combinations`

`ak.combinations` makes every pair of particles inside every event.

<img src="../img/combinations.svg" width="75%" alt="">

In [ ]:
mu1, mu2 = ak.unzip(ak.combinations(events.muon, 2))
(mu1 + mu2).mass

## NumPy and Awkward Array side by side

| | NumPy | Awkward Array |
|---|---|---|
| rows | all the same length | any length |
| arithmetic, `np.sqrt` | element by element | element by element |
| a mask | drops elements | drops particles (jagged mask) or events (flat mask) |
| reductions | `x.sum(axis=1)` | `ak.sum(x, axis=1)`, `None` for empty events |
| first item | `x[:, 0]` | `x[:, 0]`, fails on empty events |
| new tools | | `ak.num`, `ak.zip`, `ak.flatten`, `ak.combinations` |

## Now: project 4.

[project-higgs.ipynb](project-higgs.ipynb): every muon in every event.<br>Next in the school: more Awkward Array and a full HEP analysis, leading to coffea.

<img src="../img/jagged.svg" width="60%" alt="">